# 🏥 Tech Challenge — FIAP Pós 9IADT 2026 — Fase 1
## Sistema inteligente de apoio à triagem de mulheres vítimas de violência
---
> **Dados de treino**: Notificações históricas SINAN - 2010 à 2025 (`violencia_BR_25.csv`)  
> **Saída**: Probabilidade de tipo de violência e encaminhamento/exame ser necessário  
---

### 🎯 Modelos comparados neste notebook

| # | Modelo | Tipo | Justificativa |
|---|--------|------|---------------|
| 1 | **Random Forest** | Ensemble (Bagging) | robusto e estável |
| 2 | **XGBoost** | Ensemble (Boosting) | Alta performance em dados tabulares desbalanceados |
| 3 | **Logistic Regression** | Linear | Interpretável, rápido, boa base de comparação |

---

### ⚠️ Aviso Ético
> O sistema é um **apoio à decisão clínica**.
> A avaliação final é sempre responsabilidade do profissional de saúde.  
> Todos os dados tratados conforme **LGPD (Lei nº 13.709/2018)**.

---
## 1️⃣ Importar o dataset
---

In [1]:
import numpy as np
import os
import zipfile
import glob
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader

# Detecta dispositivo automaticamente
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("🚀 Usando dispositivo:", device)


def extract_all_zips(data_dir='data'):
    """Extrai todos os arquivos .zip encontrados em data_dir para a mesma pasta."""
    os.makedirs(data_dir, exist_ok=True)
    zip_paths = glob.glob(os.path.join(data_dir, '*.zip'))
    if not zip_paths:
        print("🔎 Nenhum arquivo .zip encontrado em", data_dir)
        return []
    extracted_files = []
    for zp in zip_paths:
        try:
            with zipfile.ZipFile(zp, 'r') as z:
                z.extractall(data_dir)
                extracted_files.extend(z.namelist())
            print(f"✅ Extraído: {os.path.basename(zp)}")
        except zipfile.BadZipFile:
            print(f"⚠️ Arquivo zip corrompido ou inválido: {zp}")
    return zip_paths


def _read_csv_try(path, max_cols_warn=1000):
    """Tenta ler CSV com combinações comuns de encoding e separador."""
    encodings = ['latin-1', 'utf-8', 'cp1252']
    seps = [',']
    for enc in encodings:
        for sep in seps:
            try:
                df = pd.read_csv(path, sep=sep, encoding=enc, low_memory=False)
                if df.shape[1] > max_cols_warn:
                    print(f"⚠️ {path} lido com {df.shape[1]} colunas (possível separador errado)")
                return df
            except Exception:
                continue
    df = pd.read_csv(path, low_memory=False)
    if df.shape[1] > max_cols_warn:
        print(f"⚠️ {path} lido com {df.shape[1]} colunas (pandas inferiu separador/encoding)")
    return df


class CSVDataset(Dataset):
    """Dataset PyTorch a partir de múltiplos CSVs."""
    def __init__(self, dataframes):
        # Concatena todos os DataFrames
        self.data = pd.concat(dataframes, ignore_index=True, sort=False)
        # Converte para tensor
        self.tensor = torch.tensor(self.data.fillna(0).values, dtype=torch.float32).to(device)

    def __len__(self):
        return len(self.tensor)

    def __getitem__(self, idx):
        return self.tensor[idx]


def load_all_csvs_as_dataset(data_dir='data', batch_size=32, delete_after_load=False):
    """
    Extrai zips em data_dir, carrega todos os CSVs e retorna:
      - dataset: objeto CSVDataset
      - dataloader: DataLoader pronto para treinamento
    """
    extract_all_zips(data_dir)

    csv_paths = glob.glob(os.path.join(data_dir, '**', '*.csv'), recursive=True)
    if not csv_paths:
        raise FileNotFoundError(f"Nenhum arquivo .csv encontrado em '{data_dir}'.")

    dfs = []
    for p in csv_paths:
        try:
            df = _read_csv_try(p)
            dfs.append(df)
            print(f"📥 Carregado: {os.path.basename(p)} (linhas: {len(df)}, colunas: {len(df.columns)})")
        except Exception as e:
            print(f"❌ Falha ao carregar {p}: {e}")

    dataset = CSVDataset(dfs)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    if delete_after_load:
        for p in csv_paths:
            try:
                os.remove(p)
            except Exception as e:
                print(f"⚠️ Não foi possível excluir {p}: {e}")
        print(f"🗑️ Arquivos .csv excluídos após carregamento.")

    return dataset, dataloader


Versão do PyTorch: 2.5.1+cu121
CUDA disponível: True
Nome da GPU: NVIDIA GeForce RTX 2080 SUPER
Versão CUDA usada pelo PyTorch: 12.1
